# R_only Hyperparameter Sweep

**Fixed training HPs:**
- `hidden_dim`: 16 (Cora/PubMed), 64 (Roman-empire/Squirrel)
- `lr`: 0.01 (Cora/PubMed), 0.05 (Roman-empire/Squirrel)
- `weight_decay`: 5e-4, `max_epochs`: 700, `patience`: 75, `beta`: 0.5
- **No `lambda_R`** (no CE/R tradeoff)

**Swept (8 combos):** `entropy_floor` [None,0.1] × `per_class_R` [F,T] × `band` [(-1,0),(-1.5,0.25)]

**K range: 2..8** (K=1 has no interior layers for curvature)

**Total: 1,344 runs.** Skip-existing checks for `best.pt` on disk — safe to re-run.

In [1]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
print('Installing PyTorch 2.5.1 + CUDA 12.4...')
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!sudo add-apt-repository ppa:ubuntu-toolchain-r/test -y > /dev/null 2>&1
!sudo apt-get update > /dev/null 2>&1
!sudo apt-get install --only-upgrade libstdc++6 -y > /dev/null 2>&1
!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.5.0+cu124.html
!pip install -q scikit-learn pandas matplotlib seaborn

import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

Installing PyTorch 2.5.1 + CUDA 12.4...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 110.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 55.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 10.6 MB/s 

In [2]:
# ── Cell 2: Setup repo (safe to re-run) ──────────────────────────────────────
import os, sys

if os.path.exists('src'):
    print(f'Already in repo: {os.getcwd()}')
    !git pull
elif os.path.exists('entropy-selection/src'):
    os.chdir('entropy-selection')
    print(f'Changed into repo: {os.getcwd()}')
    !git pull
else:
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
        clone_url = f'https://{token}@github.com/econci474/GDL.git'
    except Exception:
        clone_url = 'https://github.com/econci474/GDL.git'
    !git clone {clone_url} entropy-selection 2>&1 | tail -3
    os.chdir('entropy-selection')

sys.path.insert(0, os.getcwd())
print(f'Working dir: {os.getcwd()}')
assert os.path.exists('src'), 'ERROR: src/ not found!'

Cloning into 'entropy-selection'...
Working dir: /content/entropy-selection


In [4]:
# ── Cell 3: Mount Drive and link results directory (safe to re-run) ───────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

DRIVE_RESULTS = Path('/content/drive/MyDrive/GDL/r_only_sweep_results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
for d in ['runs', 'tables', 'figures', 'arrays', 'classifier_heads_R_only']:
    (DRIVE_RESULTS / d).mkdir(exist_ok=True)

local_results = Path('results')
# Remove whatever is there (real dir or broken/live symlink) then re-link
if local_results.is_symlink():
    local_results.unlink()
elif local_results.exists():
    shutil.rmtree(str(local_results))
local_results.symlink_to(DRIVE_RESULTS)

print(f'Linked results/ -> {os.readlink("results")}')
# Count already-done runs
done = list((DRIVE_RESULTS / 'classifier_heads_R_only').rglob('best.pt'))
print(f'Already completed runs (best.pt on Drive): {len(done)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Linked results/ -> /content/drive/MyDrive/GDL/r_only_sweep_results
Already completed runs (best.pt on Drive): 1368


# Hyperparameter sweep

In [ ]:
# ── Cell 4: R_only sweep (skip-existing via best.pt check) ────────────────────
import subprocess, sys, time, itertools
from pathlib import Path
import re

ROOT = Path('.')
py   = sys.executable

HOMO_DS    = ['Cora', 'PubMed']
HETERO_DS  = ['Roman-empire', 'Squirrel']
ALL_DS     = HOMO_DS + HETERO_DS
MODELS     = ['GCN', 'GAT']
K_VALUES   = list(range(2, 9))   # 2..8 (K=1 has no interior layers)
SEEDS      = [0, 1, 2]

LR_HOM       = 0.01
LR_HET       = 0.05
WEIGHT_DECAY = 5e-4
MAX_EPOCHS   = 700
PATIENCE     = 75
BETA         = 0.5
HIDDEN_HOM   = 16
HIDDEN_HET   = 64

ENTROPY_FLOORS = [None, 0.1]
PER_CLASS_R    = [False, True]
BANDS          = [(-1.0, 0.0), (-1.5, 0.25)]
entropy_combos = list(itertools.product(ENTROPY_FLOORS, PER_CLASS_R, BANDS))

# ── Skip-existing: check for best.pt in classifier_heads dir ─────────────────
# Directory name mirrors what train_gnn_entropy.py builds:
#   R_only_R1.0_smooth[_floor{f}][_perclass][_band{l}to{u}]
def expected_dir(ef, pcr, bl, bu):
    parts = ['R_only_R1.0_smooth']
    if ef is not None:
        parts.append(f'floor{ef:.2f}')
    if pcr:
        parts.append('perclass')
    if bl != -1.0 or bu != 0.0:
        parts.append(f'band{bl:.1f}to{bu:.1f}')
    return '_'.join(parts)

def is_done(ds, model, K, seed, split_id, ef, pcr, bl, bu, is_hetero):
    cfg_dir = expected_dir(ef, pcr, bl, bu)
    base = ROOT / 'results' / 'classifier_heads' / cfg_dir / ds / model / f'seed_{seed}' / f'K_{K}'
    if is_hetero:
        best = base / f'split_{split_id}' / 'best.pt'
    else:
        best = base / 'best.pt'
    return best.exists()

total = len(ALL_DS) * len(MODELS) * len(K_VALUES) * len(SEEDS) * len(entropy_combos)
run_idx, skipped, failed = 0, 0, []
t_start = time.time()

for ds in ALL_DS:
    is_hetero  = ds in HETERO_DS
    hidden_dim = HIDDEN_HET if is_hetero else HIDDEN_HOM
    lr         = LR_HET     if is_hetero else LR_HOM
    split_mode = 'first'    if is_hetero else 'auto'
    split_id   = 0

    for model in MODELS:
        for K in K_VALUES:
            for seed in SEEDS:
                for (ef, pcr, (bl, bu)) in entropy_combos:
                    run_idx += 1

                    if is_done(ds, model, K, seed, split_id, ef, pcr, bl, bu, is_hetero):
                        skipped += 1
                        continue

                    cmd = [
                        py, 'src/train_gnn_entropy.py',
                        '--dataset',      ds,
                        '--model',        model,
                        '--K',            str(K),
                        '--seed',         str(seed),
                        '--loss-type',    'R_only',
                        '--split-mode',   split_mode,
                        '--lr',           str(lr),
                        '--weight-decay', str(WEIGHT_DECAY),
                        '--max-epochs',   str(MAX_EPOCHS),
                        '--patience',     str(PATIENCE),
                        '--hidden-dim',   str(hidden_dim),
                        '--beta',         str(BETA),
                        '--band-lower',   str(bl),
                        '--band-upper',   str(bu),
                    ]
                    if ef is not None:
                        cmd += ['--entropy-floor', str(ef)]
                    if pcr:
                        cmd += ['--per-class-r']

                    done_so_far = run_idx - skipped
                    desc = (f'[{done_so_far}/{total-skipped}] '
                            f'{ds} {model} K={K} seed={seed} '
                            f'floor={ef} pcr={pcr} band=({bl},{bu})')
                    print(f'\n{"="*60}\n{desc}\n{"="*60}', flush=True)

                    t0 = time.time()
                    result = subprocess.run(cmd, cwd=ROOT)
                    elapsed = (time.time() - t0) / 60

                    if result.returncode != 0:
                        print(f'  [FAILED] exit {result.returncode} ({elapsed:.1f} min)', flush=True)
                        failed.append(desc)
                    else:
                        print(f'  [OK] {elapsed:.1f} min', flush=True)

                    elapsed_total = (time.time() - t_start) / 60
                    done_so_far = run_idx - skipped
                    if done_so_far > 0:
                        rate = elapsed_total / done_so_far
                        remaining = (total - skipped - done_so_far) * rate
                        print(f'  Elapsed: {elapsed_total:.0f} min | Est. remaining: {remaining:.0f} min ({remaining/60:.1f} h)', flush=True)

print(f'\n{"="*60}')
print(f'R_only sweep COMPLETE')
print(f'Total: {run_idx}  Skipped: {skipped}  Failed: {len(failed)}')
if failed:
    print('Failed runs:')
    for f in failed:
        print(f'  {f}')


R_only sweep COMPLETE
Total: 1344  Skipped: 1344  Failed: 0


In [ ]:
from pathlib import Path
RESULTS_DIR = Path('/content/drive/MyDrive/GDL/r_only_sweep_results')
for f in sorted(RESULTS_DIR.glob('*.csv')):
    print(f.name)

sweep_results_R_only.csv


# Hyperparameter selection per layer

In [6]:
# ── Hyperparameter selection for R_only (both band configs, per layer) ───────────
import sys, pandas as pd
from pathlib import Path

RESULTS_DIR = Path('/content/drive/MyDrive/GDL/r_only_sweep_results')  # ← adjust
MODELS      = ['GCN', 'GAT']   # ← loop over both models
SWEEP_CSV   = RESULTS_DIR / f'sweep_results_R_only.csv'

BANDS = [
    (-1.0,  0.0,  'band-1.0to0.0'),
    (-1.5,  0.25, 'band-1.5to0.25'),
]
HETERO = {'Roman-empire', 'Squirrel'}
HYPERPARAM_COLS = [
    'lr', 'weight_decay', 'patience', 'max_epochs', 'hidden_dim',
    'beta', 'lambda_r', 'entropy_floor', 'per_class_r', 'band_lower', 'band_upper',
]

df = pd.read_csv(SWEEP_CSV)
print(f"Loaded {len(df)} rows from {SWEEP_CSV.name}")

# Filter: R_only loss type only + hetero datasets split 0 only
r_df = df[df['loss_type'] == 'R_only'].copy()
hetero_mask = r_df['dataset'].isin(HETERO)
r_df = r_df[~hetero_mask | (r_df['split'] == 0)].copy()
print(f"R_only rows after hetero filter: {len(r_df)}")
r_df = r_df[r_df['K'] >= 2].copy()                        # exclude K=1 (no curvature)
print(f"R_only rows after K>=2 filter: {len(r_df)}")

for MODEL in MODELS:                                     # ← NEW outer loop
    model_df = r_df[r_df['model'] == MODEL].copy()       # ← NEW filter by model
    print(f"\n{'='*60}\nModel: {MODEL}  ({len(model_df)} rows)\n{'='*60}")
    for band_lower, band_upper, band_label in BANDS:
        band_df = model_df[
            (model_df['band_lower'] == band_lower) &
            (model_df['band_upper'] == band_upper)
        ].copy()

        if band_df.empty:
            print(f"\n[SKIP] No data for {MODEL} R_only {band_label}"); continue

        print(f"\n── {MODEL} R_only {band_label}: best per layer ──")
        results = []
        for (dataset, model, loss_type, K), grp in band_df.groupby(['dataset','model','loss_type','K']):
            agg = (grp.groupby(HYPERPARAM_COLS, dropna=False)['best_val_acc']
                      .agg(mean_val_acc='mean', n_runs='count')
                      .reset_index())
            best_row = agg.loc[agg['mean_val_acc'].idxmax()].to_dict()
            row = {'dataset': dataset, 'model': model, 'loss_type': loss_type, 'K': K,
                   'mean_val_acc': best_row['mean_val_acc'], 'n_runs': best_row['n_runs']}
            row.update({col: best_row[col] for col in HYPERPARAM_COLS})
            results.append(row)
            print(f"  {dataset}/{model}/K={K}: val_acc={best_row['mean_val_acc']:.4f}  "
                  f"lr={best_row['lr']}  wd={best_row['weight_decay']}")

        out = RESULTS_DIR / f'best_hyperparams_{MODEL}_R_only_{band_label}_per_layer.csv'
        pd.DataFrame(results).to_csv(out, index=False)
        print(f"Saved → {out.name}")

print("\nDone.")


Loaded 1368 rows from sweep_results_R_only.csv
R_only rows after hetero filter: 1368
R_only rows after K>=2 filter: 1344

Model: GCN  (672 rows)

── GCN R_only band-1.0to0.0: best per layer ──
  Cora/GCN/K=2: val_acc=0.1020  lr=0.01  wd=0.0005
  Cora/GCN/K=3: val_acc=0.1120  lr=0.01  wd=0.0005
  Cora/GCN/K=4: val_acc=0.1027  lr=0.01  wd=0.0005
  Cora/GCN/K=5: val_acc=0.1840  lr=0.01  wd=0.0005
  Cora/GCN/K=6: val_acc=0.1953  lr=0.01  wd=0.0005
  Cora/GCN/K=7: val_acc=0.1280  lr=0.01  wd=0.0005
  Cora/GCN/K=8: val_acc=0.1253  lr=0.01  wd=0.0005
  PubMed/GCN/K=2: val_acc=0.4160  lr=0.01  wd=0.0005
  PubMed/GCN/K=3: val_acc=0.2693  lr=0.01  wd=0.0005
  PubMed/GCN/K=4: val_acc=0.1960  lr=0.01  wd=0.0005
  PubMed/GCN/K=5: val_acc=0.3427  lr=0.01  wd=0.0005
  PubMed/GCN/K=6: val_acc=0.3333  lr=0.01  wd=0.0005
  PubMed/GCN/K=7: val_acc=0.2693  lr=0.01  wd=0.0005
  PubMed/GCN/K=8: val_acc=0.3240  lr=0.01  wd=0.0005
  Roman-empire/GCN/K=2: val_acc=0.0500  lr=0.05  wd=0.0005
  Roman-empire/GCN/K

# Download the checkpoints for the best model per layer onto local (one-time)

Check the models are where they are supposed to be

In [9]:
from pathlib import Path
ch = Path('results/classifier_heads_R_only')
if ch.exists():
    for p in sorted(ch.rglob('best.pt'))[:20]:
        print(p)
else:
    print('classifier_heads NOT FOUND at', ch.resolve())


results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_0/K_1/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_0/K_2/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_0/K_3/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_0/K_4/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_0/K_5/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_0/K_6/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_0/K_7/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_0/K_8/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_1/K_1/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_1/K_2/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_1/K_3/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_1/K_4/best.pt
results/classifier_heads_R_only/R_only_R1.0_smooth/C

Filter and Download

In [ ]:
import zipfile, pandas as pd
from pathlib import Path
from google.colab import files

R_RESULTS_DIR = Path('/content/drive/MyDrive/GDL/r_only_sweep_results')
HEADS_DIR     = R_RESULTS_DIR / 'classifier_heads_R_only'   # ← fixed
SEEDS         = [0, 1, 2]

HP_CSVS = [
    R_RESULTS_DIR / 'best_hyperparams_GCN_R_only_band-1.0to0.0_per_layer.csv',
    R_RESULTS_DIR / 'best_hyperparams_GCN_R_only_band-1.5to0.25_per_layer.csv',
    R_RESULTS_DIR / 'best_hyperparams_GAT_R_only_band-1.0to0.0_per_layer.csv',
    R_RESULTS_DIR / 'best_hyperparams_GAT_R_only_band-1.5to0.25_per_layer.csv',
]

def build_loss_dir(row):
    lt = row['loss_type']
    parts = [f"R{float(row['lambda_r']):.1f}", str(row.get('R_mode', 'smooth'))]
    ef = row.get('entropy_floor')
    if pd.notna(ef) and ef is not None:
        parts.append(f"floor{float(ef):.2f}")
    if pd.notna(row.get('per_class_r')) and row.get('per_class_r'):
        parts.append("perclass")
    bl, bu = float(row['band_lower']), float(row['band_upper'])
    if bl != -1.0 or bu != 0.0:
        parts.append(f"band{bl:.1f}to{bu:.1f}")
    return f"{lt}_{'_'.join(parts)}"

to_zip = []
for csv_path in HP_CSVS:
    if not csv_path.exists(): print(f"[SKIP] {csv_path.name}"); continue
    df = pd.read_csv(csv_path)
    for _, row in df.iterrows():
        loss_dir = build_loss_dir(row)
        dataset, model, K = row['dataset'], row['model'], int(row['K'])
        for seed in SEEDS:
            base = HEADS_DIR / loss_dir / dataset / model / f'seed_{seed}' / f'K_{K}'
            for candidate in [base / 'split_0' / 'best.pt', base / 'best.pt']:
                if candidate.exists():
                    arcname = Path('r_only_sweep_results') / candidate.relative_to(R_RESULTS_DIR)
                    to_zip.append((candidate, arcname)); break
            else:
                print(f"  [MISS] {loss_dir}/{dataset}/{model}/seed_{seed}/K_{K}")

for csv_path in HP_CSVS:
    if csv_path.exists():
        to_zip.append((csv_path, Path('r_only_sweep_results') / csv_path.name))

print(f"\n{len(to_zip)} files to zip")
out = Path('/content/r_only_all_K_best.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath, arcname in to_zip: zf.write(fpath, arcname)
print(f"Zip: {out.stat().st_size/1e6:.1f} MB")
files.download(str(out))




724 files to zip


/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'r_only_sweep_results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_0/K_1/best.pt'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'r_only_sweep_results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_1/K_1/best.pt'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'r_only_sweep_results/classifier_heads_R_only/R_only_R1.0_smooth/Cora/GAT/seed_2/K_1/best.pt'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'r_only_sweep_results/classifier_heads_R_only/R_only_R1.0_smooth_floor0.10/Cora/GAT/seed_0/K_2/best.pt'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate n

Zip: 3351.3 MB


/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'r_only_sweep_results/classifier_heads_R_only/R_only_R1.0_smooth_floor0.10_band-1.5to0.2/Squirrel/GCN/seed_2/K_8/split_0/best.pt'
  return self._open_to_write(zinfo, force_zip64=force_zip64)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Evaluate the best models per layer for GCN and GAT

In [10]:
import subprocess, sys
from pathlib import Path

CWD      = '/content/entropy-selection'
R_DIR    = '/content/drive/MyDrive/GDL/r_only_sweep_results'
HEADS_DIR = R_DIR + '/classifier_heads_R_only'

# Pull latest code first (gets the build_loss_dir fix)
subprocess.run(['git', 'pull'], cwd=CWD)

runs = [
    ('GCN', 'band-1.0to0.0',  f'{R_DIR}/best_hyperparams_GCN_R_only_band-1.0to0.0_per_layer.csv'),
    ('GCN', 'band-1.5to0.25', f'{R_DIR}/best_hyperparams_GCN_R_only_band-1.5to0.25_per_layer.csv'),
    ('GAT', 'band-1.0to0.0',  f'{R_DIR}/best_hyperparams_GAT_R_only_band-1.0to0.0_per_layer.csv'),
    ('GAT', 'band-1.5to0.25', f'{R_DIR}/best_hyperparams_GAT_R_only_band-1.5to0.25_per_layer.csv'),
]

for model, band, hp_csv in runs:
    out_csv = f'{R_DIR}/tables/final_results_R_only_{model}_{band}.csv'
    print(f'\n{"="*60}\n{model} R_only {band}\n{"="*60}')
    r = subprocess.run([
        sys.executable, 'src/evaluate_final.py',
        '--from-best-hyperparams',
        '--best-hyperparams-path', hp_csv,
        '--classifier-heads-dir', HEADS_DIR,
        '--results-dir', R_DIR,
        '--output-path', out_csv,
        '--seeds', '0', '1', '2',
        '--split-mode', 'first',
    ], cwd=CWD, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(f'STDERR: {r.stderr[-500:]}')



GCN R_only band-1.0to0.0
 10)
  Squirrel/GCN/K=4/seed=2/split=0 [R_only] -> test_acc=0.2257

Dataset: Squirrel
  Nodes: 2223
  Edges: 46998
  Features: 2089
  Classes: 5
  Train mask shape: (2223, 10)
  Val mask shape:   (2223, 10)
  Test mask shape:  (2223, 10)
  Squirrel/GCN/K=5/seed=0/split=0 [R_only] -> test_acc=0.2257

Dataset: Squirrel
  Nodes: 2223
  Edges: 46998
  Features: 2089
  Classes: 5
  Train mask shape: (2223, 10)
  Val mask shape:   (2223, 10)
  Test mask shape:  (2223, 10)
  Squirrel/GCN/K=5/seed=1/split=0 [R_only] -> test_acc=0.3473

Dataset: Squirrel
  Nodes: 2223
  Edges: 46998
  Features: 2089
  Classes: 5
  Train mask shape: (2223, 10)
  Val mask shape:   (2223, 10)
  Test mask shape:  (2223, 10)
  Squirrel/GCN/K=5/seed=2/split=0 [R_only] -> test_acc=0.2257

Dataset: Squirrel
  Nodes: 2223
  Edges: 46998
  Features: 2089
  Classes: 5
  Train mask shape: (2223, 10)
  Val mask shape:   (2223, 10)
  Test mask shape:  (2223, 10)
  Squirrel/GCN/K=6/seed=0/split=0 [R_

# Plot table of results for GCN and GAT

In [11]:
# Pull latest + run table for GCN and GAT
import subprocess
subprocess.run(['git', 'pull'], cwd='/content/entropy-selection')

R_DIR = '/content/drive/MyDrive/GDL/r_only_sweep_results'

for model in ['GCN', 'GAT']:
    r = subprocess.run([
        'python', 'scripts/make_summary_table.py',
        '--model', model,
        '--r-only',
        '--results-dir', f'{R_DIR}/tables', #where to read the inputs final_results csv
        '--out-dir', f'{R_DIR}/tables', #where to write the outputs png tables
    ], cwd='/content/entropy-selection', capture_output=True, text=True)
    print(r.stdout); print(r.stderr if r.returncode != 0 else '')


Saved -> /content/drive/MyDrive/GDL/r_only_sweep_results/tables/test_acc_summary_table_R_only_GCN.png


Saved -> /content/drive/MyDrive/GDL/r_only_sweep_results/tables/test_acc_summary_table_R_only_GAT.png




# Plot Figures (Entropy plots: probability and correctness)

In [17]:
# ── Fixed: extract WITH p_val_k included, then redo entropy plots ─────────────
import subprocess, sys, torch, numpy as np, pandas as pd, math
from pathlib import Path

R_RESULTS_DIR = Path('/content/drive/MyDrive/GDL/r_only_sweep_results')
HEADS_DIR     = R_RESULTS_DIR / 'classifier_heads_R_only'
ARRAYS_DIR    = R_RESULTS_DIR / 'arrays'                              # Drive backup
LOCAL_ARRAYS  = Path('/content/entropy-selection/results/arrays')     # ← where plot script reads
FIGURES_DIR   = Path('/content/entropy-selection/results/figures')
K, SEEDS, HETERO = 8, [0, 1, 2], {'Squirrel', 'Roman-empire'}
CWD = '/content/entropy-selection'; py = sys.executable

HP_CSVS = [
    ('GCN', 'band-1.0to0.0',  R_RESULTS_DIR / 'best_hyperparams_GCN_R_only_band-1.0to0.0_per_layer.csv'),
    ('GCN', 'band-1.5to0.25', R_RESULTS_DIR / 'best_hyperparams_GCN_R_only_band-1.5to0.25_per_layer.csv'),
    ('GAT', 'band-1.0to0.0',  R_RESULTS_DIR / 'best_hyperparams_GAT_R_only_band-1.0to0.0_per_layer.csv'),
    ('GAT', 'band-1.5to0.25', R_RESULTS_DIR / 'best_hyperparams_GAT_R_only_band-1.5to0.25_per_layer.csv'),
]

def build_loss_dir(row):
    lt = row['loss_type']
    parts = [f"R{float(row['lambda_r']):.1f}", str(row.get('R_mode', 'smooth'))]
    ef = row.get('entropy_floor')
    if pd.notna(ef) and ef is not None: parts.append(f"floor{float(ef):.2f}")
    if pd.notna(row.get('per_class_r')) and row.get('per_class_r'): parts.append("perclass")
    bl, bu = float(row['band_lower']), float(row['band_upper'])
    if bl != -1.0 or bu != 0.0: parts.append(f"band{bl:.1f}to{bu:.1f}")
    return f"{lt}_{'_'.join(parts)}"

def extract_nd(ckpt_path, model_name, dataset, split_idx, device):
    """Extract per-node arrays. model_name passed explicitly (DO NOT infer from path)."""
    sys.path.insert(0, CWD)
    from src.datasets import load_dataset
    from src.models import GCNNet, GATNet
    from src.utils import to_device
    data, num_classes, _ = load_dataset(dataset)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    hp   = ckpt.get('hyperparams', {})
    if split_idx is not None and data.val_mask.dim() > 1:
        data = data.clone()
        data.train_mask = data.train_mask[:, split_idx]
        data.val_mask   = data.val_mask[:, split_idx]
        data.test_mask  = data.test_mask[:, split_idx]
    labels    = data.y.numpy()
    val_mask  = data.val_mask.numpy().astype(bool)
    test_mask = data.test_mask.numpy().astype(bool)
    hidden_dim = int(hp.get('hidden_dim', 64))
    if model_name == 'GAT':
        model = GATNet(num_features=data.num_features, hidden_dim=hidden_dim,
                       num_classes=num_classes, K=K, heads=8).to(device)
    else:
        model = GCNNet(num_features=data.num_features, hidden_dim=hidden_dim,
                       num_classes=num_classes, K=K).to(device)
    model.load_state_dict(ckpt['model_state_dict']); model.eval()
    data = to_device(data, device)
    with torch.no_grad():
        _, layer_probs = model.forward_with_classifier_head(data)
    nd = {'k_list': np.arange(K + 1)}
    for k, p_k in enumerate(layer_probs):
        p = p_k.cpu().numpy(); pc = np.clip(p, 1e-10, 1.0)
        H = -np.sum(pc * np.log(pc), axis=1) / math.log(num_classes)
        e = (np.argmax(p, axis=1) != labels).astype(float)
        nd[f'p_val_{k}']  = p[val_mask];  nd[f'p_test_{k}'] = p[test_mask]
        nd[f'H_val_{k}']  = H[val_mask];  nd[f'e_val_{k}']  = e[val_mask]
        nd[f'H_test_{k}'] = H[test_mask]; nd[f'e_test_{k}'] = e[test_mask]
    return nd

def run(cmd, desc):
    print(f"\n{'─'*50}\n{desc}")
    r = subprocess.run(cmd, cwd=CWD, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-800:])
    if r.returncode != 0: print(f"  [WARNING] exit {r.returncode}\n  {r.stderr[-300:]}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ARRAYS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ARRAYS.mkdir(parents=True, exist_ok=True)

for model_name, band_label, hp_csv in HP_CSVS:
    if not hp_csv.exists(): print(f"[SKIP] {hp_csv.name}"); continue
    df = pd.read_csv(hp_csv)
    for _, hp_row in df[(df['model'] == model_name) & (df['K'] == K)].iterrows():
        dataset = hp_row['dataset']
        if dataset not in HETERO: continue   # heterophilous only
        split_idx = 0;  split_sfx = f'_split{split_idx}'
        loss_dir  = build_loss_dir(hp_row)
        print(f"\n{'='*55}\n[{band_label}] {model_name}/{dataset}\n{'='*55}")

        # ── Extract npz per seed ──────────────────────────────────────────
        for seed in SEEDS:
            npz_local = LOCAL_ARRAYS / f"{dataset}_{model_name}_K{K}_seed{seed}{split_sfx}_R_only_{band_label}_pernode.npz"
            if npz_local.exists():
                print(f"  [SKIP npz] seed={seed} already exists"); continue
            base = HEADS_DIR / loss_dir / dataset / model_name / f'seed_{seed}' / f'K_{K}' / f'split_{split_idx}'
            ckpt = base / 'best.pt'
            if not ckpt.exists(): print(f"  [SKIP ckpt] {ckpt}"); continue
            try:
                nd = extract_nd(ckpt, model_name, dataset, split_idx, device)
                np.savez(npz_local, **nd)
                print(f"  Saved local: {npz_local.name}")
                np.savez(ARRAYS_DIR / f"{dataset}_{model_name}_K{K}_seed{seed}{split_sfx}_R_only_{band_label}_pernode.npz", **nd)
            except Exception as ex:
                print(f"  [ERROR] seed={seed}: {ex}"); continue

        # ── Run entropy plots (only if at least 1 seed npz available) ────
        available = [s for s in SEEDS
                     if (LOCAL_ARRAYS / f"{dataset}_{model_name}_K{K}_seed{s}{split_sfx}_R_only_{band_label}_pernode.npz").exists()]
        if not available:
            print("  [SKIP plots] no npz available"); continue

        split_args = ['--split_idx', str(split_idx)]
        for pt in ['probability', 'correctness']:
            for seed in available:
                run([py, 'src/plot_node_entropy_vs_prob.py', '--dataset', dataset,
                     '--model', model_name, '--K', str(K), '--seed', str(seed),
                     '--split', 'val', '--plot_type', pt, '--loss-type', f'R_only_{band_label}'] + split_args,
                    f"ENTROPY {pt} seed={seed}")
            run([py, 'src/plot_node_entropy_vs_prob.py', '--dataset', dataset,
                 '--model', model_name, '--K', str(K), '--seed', '0,1,2',
                 '--split', 'val', '--plot_type', pt, '--loss-type', f'R_only_{band_label}'] + split_args,
                f"ENTROPY {pt} agg seeds=0,1,2")

print("\nDone!")



[band-1.0to0.0] GCN/Roman-empire

Dataset: Roman-empire
  Nodes: 22662
  Edges: 65854
  Features: 300
  Classes: 18
  Train mask shape: (22662, 10)
  Val mask shape:   (22662, 10)
  Test mask shape:  (22662, 10)
  Saved local: Roman-empire_GCN_K8_seed0_split0_R_only_band-1.0to0.0_pernode.npz

Dataset: Roman-empire
  Nodes: 22662
  Edges: 65854
  Features: 300
  Classes: 18
  Train mask shape: (22662, 10)
  Val mask shape:   (22662, 10)
  Test mask shape:  (22662, 10)
  Saved local: Roman-empire_GCN_K8_seed1_split0_R_only_band-1.0to0.0_pernode.npz

Dataset: Roman-empire
  Nodes: 22662
  Edges: 65854
  Features: 300
  Classes: 18
  Train mask shape: (22662, 10)
  Val mask shape:   (22662, 10)
  Test mask shape:  (22662, 10)
  Saved local: Roman-empire_GCN_K8_seed2_split0_R_only_band-1.0to0.0_pernode.npz

──────────────────────────────────────────────────
ENTROPY probability seed=0
Creating all-splits entropy vs probability plot for Roman-empire/GCN
K=8, seeds=[0], split_indices=[0], spl

# Plot Figures (separability)

In [18]:
import subprocess, sys, pandas as pd
from pathlib import Path

R_RESULTS_DIR = Path('/content/drive/MyDrive/GDL/r_only_sweep_results')
HEADS_DIR     = R_RESULTS_DIR / 'classifier_heads_R_only'
HETERO        = {'Squirrel', 'Roman-empire'}
K, CWD, py   = 8, '/content/entropy-selection', sys.executable

HP_CSVS = [
    ('GCN', 'band-1.0to0.0',  R_RESULTS_DIR / 'best_hyperparams_GCN_R_only_band-1.0to0.0_per_layer.csv'),
    ('GCN', 'band-1.5to0.25', R_RESULTS_DIR / 'best_hyperparams_GCN_R_only_band-1.5to0.25_per_layer.csv'),
    ('GAT', 'band-1.0to0.0',  R_RESULTS_DIR / 'best_hyperparams_GAT_R_only_band-1.0to0.0_per_layer.csv'),
    ('GAT', 'band-1.5to0.25', R_RESULTS_DIR / 'best_hyperparams_GAT_R_only_band-1.5to0.25_per_layer.csv'),
]

def build_loss_dir(row):
    lt = row['loss_type']
    parts = [f"R{float(row['lambda_r']):.1f}", str(row.get('R_mode','smooth'))]
    ef = row.get('entropy_floor')
    if pd.notna(ef) and ef is not None: parts.append(f"floor{float(ef):.2f}")
    if pd.notna(row.get('per_class_r')) and row.get('per_class_r'): parts.append("perclass")
    bl, bu = float(row['band_lower']), float(row['band_upper'])
    if bl != -1.0 or bu != 0.0: parts.append(f"band{bl:.1f}to{bu:.1f}")
    return f"{lt}_{'_'.join(parts)}"

for model_name, band_label, hp_csv in HP_CSVS:
    if not hp_csv.exists(): continue
    df = pd.read_csv(hp_csv)
    for _, row in df[(df['model'] == model_name) & (df['K'] == K)].iterrows():
        dataset   = row['dataset']
        loss_dir  = build_loss_dir(row)
        split_id_args = ['--split-id', '0'] if dataset in HETERO else []
        print(f"SEP seed=all  {model_name}/{dataset}  [{band_label}]  {loss_dir}")
        r = subprocess.run([
            py, 'src/separability_metrics_classifier_heads.py',
            '--dataset', dataset, '--model', model_name, '--K', str(K),
            '--seed', 'all',
            '--loss-type', loss_dir,
            '--classifier-heads-dir', str(HEADS_DIR),
        ] + split_id_args, cwd=CWD, capture_output=True, text=True)
        print(r.stdout[-500:])
        if r.returncode != 0: print('STDERR:', r.stderr[-200:])

print("Done — seed=all separability plots saved.")


SEP seed=all  GCN/Cora  [band-1.0to0.0]  R_only_R1.0_smooth_floor0.10
Processing seed 2 ...

Dataset: Cora
  Nodes: 2708
  Edges: 10556
  Features: 1433
  Classes: 7
  Train nodes: 140
  Val nodes:   500
  Test nodes:  1000

CSV saved -> results/tables/Cora_GCN_K8_seeds0_1_2_R_only_R1.0_smooth_floor0.10_separability.csv
Saved -> results/figures/Cora/GCN/K_8/Cora_GCN_k8_seed_all_R_only_R1.0_smooth_floor0.10_separability_vs_k_per_class.png

Done

SEP seed=all  GCN/PubMed  [band-1.0to0.0]  R_only_R1.0_smooth_floor0.10
ing seed 2 ...

Dataset: Pubmed
  Nodes: 19717
  Edges: 88648
  Features: 500
  Classes: 3
  Train nodes: 60
  Val nodes:   500
  Test nodes:  1000

CSV saved -> results/tables/PubMed_GCN_K8_seeds0_1_2_R_only_R1.0_smooth_floor0.10_separability.csv
Saved -> results/figures/PubMed/GCN/K_8/PubMed_GCN_k8_seed_all_R_only_R1.0_smooth_floor0.10_separability_vs_k_per_class.png

Done

SEP seed=all  GCN/Roman-empire  [band-1.0to0.0]  R_only_R1.0_smooth_floor0.10
854
  Features: 300
  

In [ ]:
import os
for path in ['/content', '/content/entropy-selection']:
    if os.path.exists(path + '/src/evaluate_final.py'):
        print(f"Found at: {path}")
        break
else:
    import subprocess
    print(subprocess.run(['find', '/content', '-name', 'evaluate_final.py', '-maxdepth', '5'],
                        capture_output=True, text=True).stdout)


Found at: /content/entropy-selection
